# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via the Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, their IDs, fields, and columns.

In [ ]:
# List all record sets with @id
record_sets = []
for record_set in metadata.record_sets:
    print(f"RecordSet name: {record_set.name}, @id: {record_set.id}")
    record_sets.append(record_set.id)
    print("  Fields and their @id values:")
    for field in record_set.fields:
        print(f"    - {field.name}: {field.id}")
    print("  Columns (if available):")
    for column in getattr(record_set, 'columns', []):
        print(f"    - {column.name}: {column.id}")
    print('-'*60)
if not record_sets:
    print("No record sets found in the metadata. Trying with dataset.records() without argument...")
    # Try to infer the available records
    try:
        for record in dataset.records():
            print("Example record:", record)
            break
    except Exception as e:
        print("No top-level records available.\nError:", e)

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# For this dataset, infer the main (and often only) record set by iterating ds.records(record_set=<id>)
# If the record sets are known, specify them explicitly. Otherwise, infer as below.
import warnings

if not record_sets:
    # This dataset might have the main record set with id 'https://api.app.sen.science/frontiers/7154140/d18d681b-431c-4fc3-b535-1e26cb261034',
    # or perhaps the title or another identifier. We'll attempt to iterate with None and print out columns.
    MAIN_RECORD_SET_ID = None
    warnings.warn("No record set found by metadata. Trying to read first record without specifying record_set.")
    try:
        records = list(dataset.records())
        df = pd.DataFrame(records)
        print("Sample columns:", df.columns.tolist())
        print(df.head())
    except Exception as e:
        print("Error extracting records:", e)
        df = pd.DataFrame()
else:
    MAIN_RECORD_SET_ID = record_sets[0]
    dataframes = {}
    # For demonstration, extract all available record sets
    for rsid in record_sets:
        try:
            records = list(dataset.records(record_set=rsid))
            df = pd.DataFrame(records)
            dataframes[rsid] = df
            print(f"Loaded {len(df)} records from {rsid}")
        except Exception as e:
            print(f"Error extracting records for {rsid}: ", e)

    print(f"Available dataframe columns for {MAIN_RECORD_SET_ID}:\n", dataframes[MAIN_RECORD_SET_ID].columns.tolist())
    display(dataframes[MAIN_RECORD_SET_ID].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# Example EDA steps
import numpy as np

chosen_df = None
if 'dataframes' in locals() and MAIN_RECORD_SET_ID:
    chosen_df = dataframes[MAIN_RECORD_SET_ID]
elif 'df' in locals() and not df.empty:
    chosen_df = df

if chosen_df is not None and len(chosen_df.columns) > 0:
    # Identify a numeric field (examples: 'MW [kDa]', 'Coverage [%]', 'Peptide count', etc)
    # Attempt to auto-detect a numeric field
    numeric_candidate_cols = [col for col in chosen_df.columns if chosen_df[col].apply(lambda v: pd.api.types.is_number(v)).sum() > 0]
    if numeric_candidate_cols:
        numeric_field = numeric_candidate_cols[0]
        print(f"Using numeric field: {numeric_field}")
        # Clean the data (make sure it's numeric)
        chosen_df[numeric_field] = pd.to_numeric(chosen_df[numeric_field], errors='coerce')
        threshold = np.percentile(chosen_df[numeric_field].dropna(), 75) # Use 75th percentile
        filtered_df = chosen_df[chosen_df[numeric_field] > threshold]
        print(f"Filtered records where {numeric_field} > {threshold:.2f} ({len(filtered_df)} records)")
        display(filtered_df.head())

        # Normalize
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Try to find a categorical/grouping field
        group_candidates = [col for col in chosen_df.columns if col != numeric_field and chosen_df[col].nunique() < 20]
        group_field = group_candidates[0] if group_candidates else None
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(f"Grouped mean {numeric_field} by {group_field}:")
            display(grouped_df.head())
        else:
            print("No suitable categorical field for grouping.")
    else:
        print("No numeric fields found for EDA.")
else:
    print("No data available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style='whitegrid')

if chosen_df is not None and len(chosen_df.columns) > 0 and 'numeric_field' in locals():
    plt.figure(figsize=(8,4))
    sns.histplot(chosen_df[numeric_field].dropna(), kde=True, bins=30)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()

    # If there's a group_field, do a boxplot
    if 'group_field' in locals() and group_field:
        plt.figure(figsize=(10,5))
        sns.boxplot(data=chosen_df, x=group_field, y=numeric_field)
        plt.xticks(rotation=45)
        plt.title(f"{numeric_field} by {group_field}")
        plt.show()
else:
    print("No data available for visualization.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- The FAIR² dataset was successfully loaded using `mlcroissant`.
- Metadata shows mass spectrometry protein data from extracellular vesicles in human mast cells.
- Numeric data (e.g., protein abundance or MW) may be extracted for further analysis; the workflow above allows you to filter and visualize this data.
- This notebook template can be extended for advanced preprocessing, custom visualizations, or machine learning analysis using the named field and record set `@id` values.